In [ ]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using Plots, BenchmarkProfiles, DataFrames, Printf, ForwardDiff, CSV

# Comparing TRAULLS with different Hessian approximations

In [ ]:
# Form results dataframes

df_gn = CSV.read("results/traulls_gn.csv", DataFrame)
df_bfgs = CSV.read("results/traulls_bfgs.csv", DataFrame)
df_sr1 = CSV.read("results/traulls_sr1.csv", DataFrame)
df_hybrid_bfgs = CSV.read("results/traulls_hybrid_bfgs.csv", DataFrame)
df_hybrid_sr1 = CSV.read("results/traulls_hybrid_sr1.csv", DataFrame);

In [ ]:
function make_traulls_table(metric, solvers)
    
    T = zeros(nrow(solvers[1]), size(solvers,1))
    
    for (j, df) in enumerate(solvers)
        for (i, row) in enumerate(eachrow(df))
            T[i,j] = row[:status] == "first_order_critical" ? row[metric] : Inf
        end
    end
    
    return T
end

## All Hessians

In [ ]:
# Comparison of Hessian approximations
solvers_df = [df_gn, df_bfgs, df_sr1, df_hybrid_bfgs, df_hybrid_sr1]
solvers = ["Gauss-Newton", "BFGS", "SR1", "HybridBFGS", "HybridSR1"];

In [ ]:
T = make_traulls_table(:elapsed_time, solvers_df)
plt_time = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :solid, :solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

T = make_traulls_table(:nouter_iter, solvers_df)
plt_nouter = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :solid, :solid, :dash, :dot], lw=1,
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of outer iterations")

T = make_traulls_table(:ninner_iter, solvers_df)
plt_ninner = performance_profile(PlotsBackend(), T, solvers, 
       palette=:tab10, linestyles = [:solid, :solid, :solid, :dash, :dot], lw=1,
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of inner iterations");

In [ ]:
plt = plot(plt_time, plt_nouter, plt_ninner; layout=3)
display(plt)

## Focus on Gauss-Newton versus structured hybrid updates (SR1 and BFGS)

In [ ]:
# Comparison of Hybrid-Hessian approaches vs. Gauss-Newon
solvers_df = [df_gn, df_hybrid_bfgs, df_hybrid_sr1]
solvers = ["Gauss-Newton", "HybridBFGS", "HybridSR1"]

# Elapsed time
T = make_traulls_table(:elapsed_time, solvers_df)
plt_traulls_time = performance_profile(PlotsBackend(), T, solvers, 
         palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

# Number of residuals evaluation
T = make_traulls_table(:neval_residual, solvers_df)
plt_traulls_neval_res = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of residuals evaluations")

# Number of gradient evaluation
T = make_traulls_table(:neval_grad, solvers_df)
plt_traulls_neval_grad = performance_profile(PlotsBackend(), T, solvers, 
       palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of gradient evaluations");

In [ ]:
plt = plot(plt_traulls_time, plt_traulls_neval_res, plt_traulls_neval_grad; layout=3)
display(plt)

## Comparing Gauss-Newton and SR1 (plain and hybrid variants) on problems with similar residual magnitude

In [ ]:
function stratified_table(solvers, metric, strat)

    T = zeros(size(strat, 1), size(solvers,1))
    
    for (j, df) in enumerate(solvers)
        zero_rows = filter(row -> row.name in zero_res, df)
        
        for (i, row) in enumerate(eachrow(zero_rows))
            T[i,j] = row[:status] == "first_order_critical" ? row[metric] : Inf
        end
    end
    
    return T
end

In [ ]:
zero_res = filter(row -> 2*row.objective < 1e-8, df_gn)[!,:name]
small_res = filter(row -> 1e-8 <= 2*row.objective < 1.0, df_gn)[!,:name]
medium_res = filter(row -> 1.0 <= 2*row.objective < 1e2, df_gn)[!,:name]
large_res = filter(row -> 1e2 <= 2*row.objective, df_gn)[!,:name]

solvers = [df_gn, df_sr1, df_hybrid_sr1]
solver_names = ["GN", "SR1", "Hybrid-SR1"]

T = stratified_table(solvers, :neval_residual, zero_res)


performance_profile(PlotsBackend(), T, solver_names, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "||rₒₚₜ||² ≤ 10⁻⁸ ")

# Comparison against other solvers

## IPOPT and Percival

In [ ]:
df_ipopt = CSV.read("results/ipopt.csv", DataFrame)
df_percival = CSV.read("results/percival.csv", DataFrame);

In [ ]:
solvers = ["TRAULLS", "IPOPT", "Percival"]

function make_performance_table(stats_traulls, stats_ipopt, stats_percival, metric)
    T = zeros(nrow(stats_traulls), 3)

    for (i, row) in enumerate(eachrow(stats_traulls))
        T[i, 1] = row[:status] == "first_order_critical" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_ipopt))
        T[i, 2] = row[:status] == "first_order" ? row[metric] : Inf
    end

    for (i, row) in enumerate(eachrow(stats_percival))
        T[i, 3] = row[:status] == "first_order" ? row[metric] : Inf
    end

    return T
end

performance_table(metric) = make_performance_table(df_hybrid_sr1, df_ipopt, df_percival, metric)

In [ ]:
# Elapsed time
T = performance_table(:elapsed_time)
plt_elapsed_time = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

# Residual evaluation
T = performance_table(:neval_residual)
plt_neval_res = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of residuals evaluations")

# Gradient evaluations
T = performance_table(:neval_grad)
plt_neval_grad = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of gradient evaluations");

In [ ]:
plt = plot(plt_elapsed_time, plt_neval_res, plt_neval_grad; layout=3)
display(plt)

## Percival and IPOPT with L-BFGS

In [ ]:
df_ipopt_lbfgs = CSV.read("results/ipopt-lbfgs.csv", DataFrame);

In [ ]:
solvers = ["TRAULLS", "IPOPT-LBFGS", "Percival"]


performance_table(metric) = make_performance_table(df_hybrid_sr1, df_ipopt_lbfgs, df_percival, metric)

In [ ]:
# Elapsed time
T = performance_table(:elapsed_time)
plt_elapsed_time = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

# Residual evaluation
T = performance_table(:neval_residual)
plt_neval_res = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of residuals evaluations")

# Gradient evaluations
T = performance_table(:neval_grad)
plt_neval_grad = performance_profile(PlotsBackend(), T, solvers, 
        palette=:tab10, linestyles = [:solid, :dash, :dot], lw=1, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of gradient evaluations");

In [ ]:
plt = plot(plt_elapsed_time, plt_neval_res, plt_neval_grad; layout=3)
display(plt)